# Part 4: Recommendation Engine + Final Report

**Pipeline position** (from `new_proposed_pipeline`):
```
...  ->  Explainable AI Module (Part 3)  ->  Recommendation Engine  ->  Final Report
                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                                       THIS NOTEBOOK
```

This is the last part -- it loads everything already built and produces the one
function a real deployment (API/app) would call.

**Inputs needed** (upload all of these when prompted):
- `pipeline.py` (from Part 1 -- needed for `extract_features()` on new resumes)
- `rf_model.joblib`, `model_config.joblib` (from Part 2)

**Output:** `build_final_report(candidate_name, resume_text, jd_text)` -- the
complete end-to-end function.


## 1. Setup -- load the already-trained model (no retraining here)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()  # upload pipeline.py, rf_model.joblib, model_config.joblib
except ImportError:
    pass

!pip install rapidfuzz shap sentence-transformers scikit-learn pdfplumber -q


In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

import pipeline as p
import shap

model = joblib.load("rf_model.joblib")
config = joblib.load("model_config.joblib")

FEATURES = config["features"]
CONFIDENCE_THRESHOLD = config["confidence_threshold"]
EXPERIENCE_MEDIAN = config["experience_median"]

explainer = shap.TreeExplainer(model)

print("Loaded model. Classes:", model.classes_)
print("Features:", FEATURES)
print("Confidence threshold:", CONFIDENCE_THRESHOLD)


## 2. Recommendation Engine

For every missing skill, we suggest a specific, well-known certification or course
where one exists, and fall back to a generic-but-still-actionable suggestion
otherwise. This mapping is intentionally curated (not exhaustive) -- covering the
most common skills across your dataset's domains (Finance, HR, Marketing, Data
Science, DevOps, Cyber Security, UX/UI, Sales).

In [ ]:
LEARNING_RESOURCES = {
    # Programming / Software Development
    "python": "Python for Everybody (Coursera) or a portfolio project using Python",
    "java": "Oracle Certified Professional: Java Programmer",
    "cplusplus": "A DSA-focused C++ course (e.g. GeeksforGeeks / Coursera)",
    "javascript": "JavaScript Algorithms and Data Structures (freeCodeCamp)",
    "react": "Meta Front-End Developer Certificate (Coursera)",
    "django": "Django for Everybody (Coursera)",
    "microservices": "Microservices Architecture course (e.g. Coursera / Udemy)",
    # Cloud / DevOps
    "aws": "AWS Certified Solutions Architect - Associate",
    "azure": "Microsoft Certified: Azure Fundamentals (AZ-900)",
    "gcp": "Google Cloud Associate Cloud Engineer",
    "docker": "Docker Certified Associate",
    "kubernetes": "Certified Kubernetes Administrator (CKA)",
    "terraform": "HashiCorp Certified: Terraform Associate",
    "prometheus": "Prometheus Certified Associate (PCA)",
    # Data Science / ML
    "machine learning": "Machine Learning Specialization (Andrew Ng, Coursera)",
    "tensorflow": "TensorFlow Developer Certificate",
    "pytorch": "PyTorch for Deep Learning (Udacity/Coursera)",
    "sql": "SQL for Data Science (Coursera) or a hands-on project with a real dataset",
    "power bi": "Microsoft Certified: Power BI Data Analyst Associate",
    "tableau": "Tableau Desktop Specialist Certification",
    "airflow": "Apache Airflow fundamentals course",
    # Finance
    "cfa": "CFA Program (Level I)",
    "sap": "SAP Certified Application Associate",
    "gaap": "US GAAP fundamentals course (e.g. AICPA resources)",
    "ifrs": "IFRS Certificate (ACCA / ICAEW)",
    "financial modeling": "Financial Modeling & Valuation Analyst (FMVA)",
    "quickbooks": "QuickBooks Certified User exam",
    "valuation": "Business Valuation certificate course (CFI)",
    # HR
    "hrms": "SHRM Certified Professional (SHRM-CP)",
    "workday": "Workday HCM certification (via a Workday partner)",
    "payroll": "Certified Payroll Professional (CPP)",
    # Marketing
    "seo": "Google SEO Fundamentals (via Google Skillshop)",
    "google analytics": "Google Analytics Individual Qualification (GAIQ)",
    "hubspot": "HubSpot Content Marketing Certification",
    "google ads": "Google Ads Certification (Google Skillshop)",
    # Product / Project Management
    "agile": "Certified ScrumMaster (CSM)",
    "jira": "Atlassian Jira Fundamentals course",
    "product management": "Product Management Certificate (Product School / Coursera)",
    # Cyber Security
    "cissp": "CISSP Certification (ISC2)",
    "splunk": "Splunk Core Certified User",
    "penetration testing": "CompTIA PenTest+ or eJPT",
    "ceh": "Certified Ethical Hacker (CEH)",
    # UX / UI
    "figma": "Figma's own Academy courses (free)",
    "wireframing": "Google UX Design Certificate (Coursera)",
    "design systems": "Design Systems course (Interaction Design Foundation)",
    # Sales
    "salesforce": "Salesforce Certified Administrator",
    "crm": "HubSpot CRM Certification (free)",
}


def suggest_learning_resources(missing_skills: list, max_suggestions: int = 5) -> list:
    """
    For each missing skill, return a specific certification/course suggestion
    if we have one mapped; otherwise fall back to a generic but still actionable
    recommendation. Capped at max_suggestions so the report stays readable --
    showing 15 missing skills at once isn't useful feedback, it's a wall of text.
    """
    suggestions = []
    for skill in missing_skills[:max_suggestions]:
        resource = LEARNING_RESOURCES.get(
            skill, f"An online course or hands-on project in {skill.replace('_', ' ').title()}"
        )
        suggestions.append({"skill": skill, "resource": resource})
    return suggestions


# quick test
suggest_learning_resources(["kubernetes", "sap", "splunk", "some_niche_tool"])


## 3. Final Report Generator

This is the single function your API/app will actually call: resume text + JD text
in, complete structured report out -- matching the "Final Output" section of your
proposal (Candidate Name, Overall Match, Matched/Missing Skills, Experience
Analysis, Domain Analysis, Semantic Similarity, Prediction Explanation,
Improvement Recommendations).

In [ ]:
FEATURE_LABELS = {
    "skill_match_score": "Skill Match",
    "matched_skill_count": "Number of Matched Skills",
    "missing_skill_count": "Number of Missing Skills",
    "experience_match_score": "Experience Match",
    "resume_experience_mentioned": "Resume States Experience",
    "jd_experience_mentioned": "JD States Required Experience",
    "semantic_similarity_score": "Overall Semantic Fit",
    "domain_match": "Domain Match",
}


def build_final_report(candidate_name: str, resume_text: str, jd_text: str) -> dict:
    """
    Full end-to-end: resume + JD text -> complete candidate report.
    This is the ONE function a deployment (FastAPI endpoint, Streamlit app)
    needs to call.
    """
    extracted = p.extract_features(resume_text, jd_text)

    row_features = {
        "skill_match_score": extracted["skill_match_score"],
        "matched_skill_count": extracted["matched_skill_count"],
        "missing_skill_count": extracted["missing_skill_count"],
        "experience_match_score": (
            extracted["experience_match_score"]
            if not pd.isna(extracted["experience_match_score"])
            else EXPERIENCE_MEDIAN
        ),
        "resume_experience_mentioned": extracted["resume_experience_mentioned"],
        "jd_experience_mentioned": extracted["jd_experience_mentioned"],
        "semantic_similarity_score": extracted["semantic_similarity_score"],
        "domain_match": extracted["domain_match"],
    }
    X_row = pd.DataFrame([row_features])[FEATURES]

    proba = model.predict_proba(X_row)[0]
    pred_class = model.classes_[np.argmax(proba)]
    confidence = float(np.max(proba))
    needs_review = confidence < CONFIDENCE_THRESHOLD

    sv = explainer.shap_values(X_row)
    class_idx = list(model.classes_).index(pred_class)
    contributions = sv[0, :, class_idx]
    impact = sorted(zip(FEATURES, contributions), key=lambda t: -abs(t[1]))
    reasons = [{
        "factor": FEATURE_LABELS[feat],
        "value": round(float(X_row[feat].values[0]), 2),
        "direction": "supported" if contrib > 0 else "worked against",
        "impact": round(float(contrib), 3),
    } for feat, contrib in impact[:4]]

    recommendations = suggest_learning_resources(extracted["missing_skills"])

    return {
        "candidate_name": candidate_name,
        "prediction": pred_class,
        "confidence": round(confidence, 2),
        "needs_human_review": needs_review,
        "matched_skills": extracted["matched_skills"],
        "missing_skills": extracted["missing_skills"],
        "resume_experience": extracted["resume_experience"],
        "jd_experience": extracted["jd_experience"],
        "experience_match_score": round(row_features["experience_match_score"], 2),
        "resume_domain": extracted["resume_domain"],
        "jd_domain": extracted["jd_domain"],
        "domain_match": bool(extracted["domain_match"]),
        "semantic_similarity_score": extracted["semantic_similarity_score"],
        "reasons": reasons,
        "recommendations": recommendations,
    }


### Pretty-print helper (for a human-readable version of the same report)

In [ ]:
def print_final_report(report: dict):
    print("=" * 60)
    print(f"CANDIDATE REPORT: {report['candidate_name']}")
    print("=" * 60)
    print(f"Prediction: {report['prediction'].upper()}  (confidence: {report['confidence']:.0%})")
    if report["needs_human_review"]:
        print("\u26a0\ufe0f  LOW CONFIDENCE -- recommended for human review")
    print()
    print(f"Matched Skills ({len(report['matched_skills'])}):", ", ".join(report["matched_skills"]) or "None")
    print(f"Missing Skills ({len(report['missing_skills'])}): ", ", ".join(report["missing_skills"]) or "None")
    print()
    print(f"Experience: {report['resume_experience']} yrs (candidate) vs "
          f"{report['jd_experience']} yrs (required) -- match score {report['experience_match_score']}")
    print(f"Domain: {report['resume_domain']} (candidate) vs {report['jd_domain']} (JD) "
          f"-- {'Match' if report['domain_match'] else 'No Match'}")
    print(f"Semantic Similarity: {report['semantic_similarity_score']}")
    print()
    print("Why this prediction:")
    for r in report["reasons"]:
        print(f"  - {r['factor']} ({r['value']}) {r['direction']} this prediction (impact: {r['impact']:+.3f})")
    print()
    if report["recommendations"]:
        print("Recommendations to close the gap:")
        for rec in report["recommendations"]:
            print(f"  - Learn {rec['skill']}: {rec['resource']}")
    else:
        print("No skill gaps detected -- strong match on required skills.")
    print("=" * 60)


## 4. End-to-end test: new resume + JD → complete final report

In [ ]:
resume_text = """
John Doe
Data Scientist with 5 years of experience in machine learning and Python.
Skills: Python, SQL, Machine Learning, TensorFlow, AWS, Docker
Education: B.Tech in Computer Science
"""

jd_text = """
Job Title: Machine Learning Engineer
Experience: 4 years required
Required Skills: Python, TensorFlow, AWS, Kubernetes, SQL
"""

report = build_final_report("John Doe", resume_text, jd_text)
print_final_report(report)

print()
print("Raw JSON (what an API endpoint would actually return):")
print(json.dumps(report, indent=2))


### Same thing, starting from an actual uploaded PDF

In [ ]:
# from pipeline import ScannedPDFError, CorruptPDFError
#
# try:
#     resume_text_from_pdf = p.extract_text_from_pdf("resume.pdf")
#     report = build_final_report("Candidate Name", resume_text_from_pdf, jd_text)
#     print_final_report(report)
# except ScannedPDFError as e:
#     print(f"Could not process this resume: {e}")
# except CorruptPDFError as e:
#     print(f"File error: {e}")


## 5. What's left after this notebook

Everything in the proposed architecture is now built EXCEPT deployment:

```
Feature Generation           ✅ pipeline.py
Candidate Matching Engine    ✅ Matching_Engine_and_Explainability.ipynb
Explainable AI Module        ✅ Matching_Engine_and_Explainability.ipynb
Recommendation Engine        ✅ this notebook
Final Report                 ✅ this notebook (build_final_report)
------------------------------------------------------------
Deployment (API + demo UI)   ⬜ not built yet
```

The one function a deployment needs to call is `build_final_report(candidate_name,
resume_text, jd_text)` -- everything else in this notebook is either setup for
that function or a way to test it.